# nb06 — GPU Scale: HuBERT Featurizer
> **When CPU accuracy isn't enough, use a frozen HuBERT featurizer.**  
> The `medium` tier uses a pre-exported HuBERT-ONNX featurizer (~90 M parameters)
> and a small FFN head. The `large` tier adds a biGRU-256 head. Both require GPU.

---

## When to use GPU tiers

| Situation | Recommendation |
|-----------|---------------|
| RPi 4 / laptop, need best F1 | Start with `medium` |
| Server / workstation, latency not critical | Use `large` |
| Want portable featurizer for CPU deployment | See nb07 + nb08 |
| CPU-only deployment required | Use nb05 tiers |

## HuBERT as featurizer

- HuBERT is **frozen** (no fine-tuning) — only the classification head is trained.
- This dramatically reduces training time vs full fine-tuning.
- The 768-d frame-level embeddings from HuBERT's 6th layer carry rich
  phonetic information that generalises across speakers and conditions.
- **RTF penalty**: HuBERT processes audio ~100× slower than MFCC per second.
  On a modern CPU it runs at RTF ≈ 0.5–2.0 — not suitable for always-on.
  On GPU (T4): RTF ≈ 0.02–0.05.

## GPU requirements

| Tier | Min VRAM | Notes |
|------|----------|-------|
| `medium` | 4 GB | HuBERT-ONNX featurizer, FFN head |
| `large` | 8 GB | HuBERT PyTorch featurizer, biGRU-256 head |

The notebook checks available VRAM and skips `large` automatically if < 8 GB.

In [ ]:
import os

# ── Core ──────────────────────────────────────────────────────────────────────
WAKE_WORD         = os.environ.get("WAKE_WORD",         "hey jarvis")
OUTPUT_DIR        = os.environ.get("OUTPUT_DIR",        "./ww_output")
DEVICE            = os.environ.get("DEVICE",            "cuda")
SEED              = int(os.environ.get("SEED",          "42"))

# ── Model ─────────────────────────────────────────────────────────────────────
TIER              = os.environ.get("TIER",              "medium")
EPOCHS            = int(os.environ.get("EPOCHS",        "30"))
BATCH_SIZE        = int(os.environ.get("BATCH_SIZE",    "32"))
TRAIN_LARGE       = os.environ.get("TRAIN_LARGE",       "auto")  # auto/true/false

# ── Dataset ───────────────────────────────────────────────────────────────────
N_POSITIVE        = int(os.environ.get("N_POSITIVE",    "500"))
LANG              = os.environ.get("LANG",              "en")
ADVERSARIAL       = os.environ.get("ADVERSARIAL",       "true").lower() == "true"
DOWNLOAD_AUGMENT  = os.environ.get("DOWNLOAD_AUGMENT",  "true").lower() == "true"
REUSE_DATASET     = os.environ.get("REUSE_DATASET",     "true").lower() == "true"
CUSTOM_TRAIN_CSV  = os.environ.get("CUSTOM_TRAIN_CSV",  "")
CUSTOM_TEST_CSV   = os.environ.get("CUSTOM_TEST_CSV",   "")
SKIP_COMPLETED    = os.environ.get("SKIP_COMPLETED",    "true").lower() == "true"

# ── MLflow (optional) ────────────────────────────────────────────────────────
MLFLOW_URI        = os.environ.get("MLFLOW_URI",        "")
MLFLOW_SECRET     = os.environ.get("MLFLOW_SECRET",     "MLFLOW_TOKEN")

print(f"Tier     : {TIER!r} (primary)")
print(f"Epochs   : {EPOCHS}  |  Batch: {BATCH_SIZE}")
print(f"Wake word: {WAKE_WORD!r}")
print(f"Large tier: {TRAIN_LARGE!r} (auto = train if VRAM >= 8 GB)")

In [ ]:
import subprocess, sys, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "pandas", "librosa", "onnx", "onnxruntime", "click", "tqdm")
_pip("transformers", "accelerate",
     "ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "ovos-vad-plugin-silero", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)

import torch

# GPU check — this notebook requires CUDA
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available. This notebook requires a GPU.\n"
        "For CPU-only training, use nb05_embedded.ipynb instead."
    )

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU    : {torch.cuda.get_device_name(0)}  |  VRAM: {vram_gb:.1f} GB")

# Decide whether to train large tier
if TRAIN_LARGE == "auto":
    _do_large = vram_gb >= 8.0
else:
    _do_large = TRAIN_LARGE.lower() == "true"
print(f"Train large tier: {_do_large} ({'VRAM >= 8 GB' if _do_large else 'VRAM < 8 GB or disabled'})")

if _platform == "kaggle" and MLFLOW_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(MLFLOW_SECRET)
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
    except Exception as e:
        print(f"MLflow secret not found: {e}")
if MLFLOW_URI:
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URI

In [ ]:
import shutil
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
# HuBERT model cache needs ~1.5 GB extra
assert free_gb > 5, f"Only {free_gb:.1f} GB free — need at least 5 GB (HuBERT cache ~1.5 GB)."
print(f"Disk free: {free_gb:.1f} GB")

_aug_kwargs_full = {}

if CUSTOM_TRAIN_CSV:
    import random, csv as _csv
    from ww_trainer.utils import read_dataset_csv
    train_csv = Path(CUSTOM_TRAIN_CSV)
    test_csv  = Path(CUSTOM_TEST_CSV) if CUSTOM_TEST_CSV else None
    if test_csv is None:
        split_dir = Path(OUTPUT_DIR) / "dataset_split"
        split_dir.mkdir(parents=True, exist_ok=True)
        split_train = split_dir / "train.csv"
        split_test  = split_dir / "test.csv"
        if not split_train.exists():
            rows = read_dataset_csv(train_csv)
            random.seed(SEED); random.shuffle(rows)
            cut = int(len(rows) * 0.8)
            for p, rs in [(split_train, rows[:cut]), (split_test, rows[cut:])]:
                with open(p, "w", newline="") as f:
                    _csv.writer(f).writerows(rs)
        train_csv, test_csv = split_train, split_test
else:
    from ww_trainer.datagen import DatagenConfig, run_datagen_pipeline, DatagenResult, normalize_wake_word
    dataset_dir = Path(OUTPUT_DIR) / "dataset"
    _train_csv_check = dataset_dir / "train" / "metadata.csv"
    if REUSE_DATASET and _train_csv_check.exists():
        slug = normalize_wake_word(WAKE_WORD)
        _dr = DatagenResult(
            train_csv=dataset_dir / "train" / "metadata.csv",
            test_csv=dataset_dir / "test" / "metadata.csv",
            positives_dir=dataset_dir / slug / "positives",
            negatives_dir=dataset_dir / slug / "negatives",
            bg_noise_dir=dataset_dir / "augmentation" / "bg_noise",
            music_dir=dataset_dir / "augmentation" / "music",
            rir_dir=dataset_dir / "augmentation" / "rir",
        )
        print(f"Reusing dataset at {dataset_dir}")
    else:
        _dr = run_datagen_pipeline(DatagenConfig(
            wake_word=WAKE_WORD, output_dir=dataset_dir,
            n_positive=N_POSITIVE, lang=LANG,
            adversarial=ADVERSARIAL, vad_trim=True,
            download_augmentation=DOWNLOAD_AUGMENT, seed=SEED,
        ))
    train_csv, test_csv = _dr.train_csv, _dr.test_csv
    for attr, key in [("bg_noise_dir", "bg_noise_folder"),
                      ("music_dir", "music_folder"),
                      ("rir_dir", "rir_folder")]:
        d = getattr(_dr, attr, None)
        if d and Path(d).exists():
            _aug_kwargs_full[key] = str(d)

print(f"train_csv: {train_csv}")
print(f"test_csv : {test_csv}")

In [ ]:
import json, time
from pathlib import Path
from ww_trainer.quickstart import train_from_wakeword

# ── Train medium tier (HuBERT-ONNX featurizer + FFN head) ─────────────────────
# HuBERT is loaded from HuggingFace (facebook/hubert-base-ls960) and exported
# to ONNX on first run — cached locally afterwards.

results_dir = Path(OUTPUT_DIR) / "gpu_results"
results_dir.mkdir(parents=True, exist_ok=True)
all_results = []

def _run_tier(tier, extra_kw=None):
    result_file = results_dir / f"{tier}.json"
    model_subdir = Path(OUTPUT_DIR) / "models" / tier
    print(f"\n{'='*60}\nTier: {tier!r}")
    if SKIP_COMPLETED and result_file.exists():
        saved = json.loads(result_file.read_text())
        print(f"  SKIP: F1={saved.get('f1', 0):.4f}")
        all_results.append(saved)
        return
    kw = dict(_aug_kwargs_full)
    if extra_kw:
        kw.update(extra_kw)
    t0 = time.time()
    try:
        r = train_from_wakeword(
            WAKE_WORD, str(model_subdir),
            tier=tier, epochs=EPOCHS, batch_size=BATCH_SIZE,
            device=DEVICE, seed=SEED, reuse_dataset=True,
            **kw,
        )
        elapsed = time.time() - t0
        row = {
            "tier": tier,
            "f1": r.metrics.get("f1", 0.0),
            "precision": r.metrics.get("precision", 0.0),
            "recall": r.metrics.get("recall", 0.0),
            "elapsed_s": round(elapsed, 1),
            "head_onnx": str(r.best_onnx_path) if r.best_onnx_path else "",
            "feat_onnx": str(model_subdir / "model" / "best_f1_featurizer.onnx"),
            "status": "ok",
        }
        print(f"  DONE: F1={row['f1']:.4f}  ({elapsed:.0f}s)")
    except Exception as exc:
        elapsed = time.time() - t0
        row = {
            "tier": tier, "f1": 0.0, "precision": 0.0, "recall": 0.0,
            "elapsed_s": round(elapsed, 1), "head_onnx": "", "feat_onnx": "",
            "status": f"error: {exc}",
        }
        print(f"  ERROR: {exc}")
    result_file.write_text(json.dumps(row, indent=2))
    all_results.append(row)


_run_tier("medium")
print(f"\nMedium tier done.")

In [ ]:
import torch

# ── Train large tier (HuBERT PyTorch + biGRU-256) — optional ─────────────────
# Requires >= 8 GB VRAM. Skipped automatically if VRAM is insufficient.
# Large tier uses a bidirectional GRU on top of full HuBERT features —
# higher accuracy but ~3× slower than medium.

if _do_large:
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB — training large tier")
    _run_tier("large")
else:
    vram_gb_now = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Skipping large tier (VRAM={vram_gb_now:.1f} GB < 8 GB or TRAIN_LARGE=false).")
    print("Set TRAIN_LARGE=true to force.")

# Also load small results from disk if available (for comparison in Cell 7)
_small_result_file = Path(OUTPUT_DIR) / "gpu_results" / "small.json"
# Check nb05 output location too
_nb05_small = Path(OUTPUT_DIR) / "embedded_results" / "small.json"
if not _small_result_file.exists() and _nb05_small.exists():
    import shutil
    shutil.copy(_nb05_small, _small_result_file)
    print(f"Copied small-tier results from nb05: {_nb05_small}")

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ── Compare: medium vs large vs small ─────────────────────────────────────────

# Load all available results
_compare_tiers = ["small", "medium", "large"]
compare_rows = []
for tier in _compare_tiers:
    for search_dir in [
        Path(OUTPUT_DIR) / "gpu_results",
        Path(OUTPUT_DIR) / "embedded_results",
        Path(OUTPUT_DIR) / "micro_results",
    ]:
        rf = search_dir / f"{tier}.json"
        if rf.exists():
            r = json.loads(rf.read_text())
            r["source"] = search_dir.name
            compare_rows.append(r)
            break

# Also add current session results
current_keys = {r["tier"] for r in compare_rows}
for r in all_results:
    if r["tier"] not in current_keys:
        r2 = dict(r); r2["source"] = "current"
        compare_rows.append(r2)

df_cmp = pd.DataFrame(compare_rows)
_cols = ["tier", "f1", "precision", "recall", "elapsed_s", "status"]
_cols = [c for c in _cols if c in df_cmp.columns]
print("Tier comparison (F1 / EER):")
print(df_cmp[_cols].sort_values("f1", ascending=False).to_string(index=False))

df_ok = df_cmp[df_cmp["status"] == "ok"].copy()
if not df_ok.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = {"small": "coral", "medium": "steelblue", "large": "seagreen"}
    bars = ax.bar(
        df_ok["tier"], df_ok["f1"],
        color=[colors.get(t, "gray") for t in df_ok["tier"]],
        edgecolor="white"
    )
    ax.set_ylabel("F1")
    ax.set_title(f"F1 comparison — {WAKE_WORD!r}")
    ax.set_ylim(0, 1.05)
    for bar, f1 in zip(bars, df_ok["f1"]):
        ax.text(bar.get_x() + bar.get_width()/2, f1 + 0.01, f"{f1:.3f}",
                ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    cmp_path = str(Path(OUTPUT_DIR) / "gpu_comparison.png")
    plt.savefig(cmp_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Comparison plot saved: {cmp_path}")

In [ ]:
import numpy as np
import csv
import torchaudio
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import onnxruntime as ort

# ── t-SNE of HuBERT embeddings ────────────────────────────────────────────────
# Load up to 300 test samples, extract HuBERT embeddings, project with t-SNE.
# Well-separated clusters = HuBERT is a useful featurizer for this wake word.

from sklearn.manifold import TSNE

medium_row = next((r for r in all_results if r["tier"] == "medium" and r["status"] == "ok"), None)

if medium_row is None or not Path(medium_row.get("feat_onnx", "")).exists():
    print("Medium tier featurizer not available — skipping t-SNE.")
else:
    print("Extracting HuBERT embeddings for t-SNE...")
    sess = ort.InferenceSession(
        medium_row["feat_onnx"], providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    )
    in_name = sess.get_inputs()[0].name

    wavs_for_tsne, labels_for_tsne = [], []
    with open(test_csv) as f:
        for row in csv.reader(f):
            if len(row) < 2 or not Path(row[0]).exists():
                continue
            wav, sr = torchaudio.load(row[0])
            if sr != 16000:
                wav = torchaudio.functional.resample(wav, sr, 16000)
            wavs_for_tsne.append(wav.mean(0).numpy().astype(np.float32))
            labels_for_tsne.append(int(row[1].strip()))
            if len(wavs_for_tsne) >= 300:
                break

    embs = []
    for wav in wavs_for_tsne:
        out = sess.run(None, {in_name: wav[np.newaxis, :]})[0]
        embs.append(out.mean(axis=1).ravel() if out.ndim == 3 else out.ravel())
    embs = np.array(embs)
    labels_arr = np.array(labels_for_tsne)

    print(f"  {len(embs)} embeddings, dim={embs.shape[1]}. Running t-SNE...")
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=min(30, len(embs)//4),
                n_iter=1000, verbose=0)
    coords = tsne.fit_transform(embs)

    fig, ax = plt.subplots(figsize=(8, 6))
    for lbl, color, name in [(1, "steelblue", "wake"), (0, "coral", "non-wake")]:
        mask = labels_arr == lbl
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=color, s=20, alpha=0.6, label=f"{name} (n={mask.sum()})")
    ax.set_title(f"t-SNE of HuBERT embeddings (medium tier) — {WAKE_WORD!r}")
    ax.set_xlabel("t-SNE dim 1")
    ax.set_ylabel("t-SNE dim 2")
    ax.legend(fontsize=9)
    plt.tight_layout()
    tsne_path = str(Path(OUTPUT_DIR) / "hubert_tsne.png")
    plt.savefig(tsne_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"t-SNE plot saved: {tsne_path}")

In [ ]:
from pathlib import Path

# ── ONNX export + size check ──────────────────────────────────────────────────
print("ONNX file audit:")
for row in all_results:
    tier = row["tier"]
    for label, path_str in [("feat", row.get("feat_onnx","")), ("head", row.get("head_onnx",""))]:
        p = Path(path_str) if path_str else None
        if p and p.exists():
            print(f"  OK   {tier:10s} {label}: {p.name}  ({p.stat().st_size/1024/1024:.1f} MB)")
        else:
            print(f"  MISS {tier:10s} {label}: {path_str}")

print()
print("Note: HuBERT featurizer ONNX is large (~300–400 MB).")
print("For a portable featurizer that fits in <20 MB, see nb07 (TinyHuBERT distillation).")

In [ ]:
import time
import numpy as np
import csv
import torchaudio
from pathlib import Path
from ww_trainer.inference import OnnxWakeWordInferencer

# ── RTF benchmark: HuBERT vs MFCC ─────────────────────────────────────────────
# Shows the RTF penalty of HuBERT relative to MFCC.
# Expected: HuBERT ~100× slower per second of audio.

import numpy as np
dummy_1s = np.random.randn(16000).astype(np.float32)

print("RTF benchmark (20 runs each on 1s of audio):")
for row in all_results:
    if row["status"] != "ok":
        continue
    feat_p = Path(row["feat_onnx"])
    head_p = Path(row["head_onnx"])
    if not feat_p.exists() or not head_p.exists():
        continue
    inf = OnnxWakeWordInferencer(str(feat_p), str(head_p))
    # Warm-up
    for _ in range(2): inf.infer(dummy_1s)
    t0 = time.perf_counter()
    for _ in range(20): inf.infer(dummy_1s)
    lat_ms = (time.perf_counter() - t0) / 20 * 1000
    rtf = lat_ms / 1000
    print(f"  {row['tier']:15s}: {lat_ms:.1f} ms  RTF={rtf:.4f}")

print()
print("Inference test on a positive sample:")
best_row = sorted([r for r in all_results if r["status"] == "ok"],
                  key=lambda r: r.get("f1", 0), reverse=True)
best_row = best_row[0] if best_row else None
if best_row and Path(best_row.get("feat_onnx", "")).exists():
    inf = OnnxWakeWordInferencer(best_row["feat_onnx"], best_row["head_onnx"])
    _pos_path = None
    with open(test_csv) as f:
        for row in csv.reader(f):
            if len(row) >= 2 and row[1].strip() == "1" and Path(row[0]).exists():
                _pos_path = row[0]; break
    if _pos_path:
        wav, sr = torchaudio.load(_pos_path)
        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)
        score = inf.infer(wav.mean(0).numpy().astype(np.float32))
        print(f"  {best_row['tier']!r}: score={score:.4f}  ({'PASS' if score > 0.5 else 'LOW'})")

print()
print("=" * 60)
if best_row:
    print(f"Best tier: {best_row['tier']!r}  F1={best_row['f1']:.4f}")
    print(f"  .venv/bin/python scripts/eval/test_wakeword.py \\")
    print(f"      --featurizer {best_row['feat_onnx']} \\")
    print(f"      --model      {best_row['head_onnx']} \\")
    print(f"      --audio      sample.wav")
print("For a CPU-deployable version, run nb07 + nb08.")
print("=" * 60)